# 00 — Cycling Counter Data Extraction

This notebook is the ingestion/extraction stage for the cycling-counter workflow.

It scans the Transport Victoria Bicycle Volume and Speed yearly archives, extracts the requested counters, validates site IDs and dates, and produces reproducible processed datasets for later quality assessment and analysis.

## Outputs

Generated files are written under:

`data/processed/cycling_counters/extraction/`

The raw source archives are never modified.

## Current counters

- Wellington Street — 32493
- Albert Street — 9077
- Moorabool Street — 34687, linked IDs 64644 and 64645
- Heidelberg Road — 40004 and 40005

In [1]:
from pathlib import Path
from zipfile import ZipFile, BadZipFile
import re
import pandas as pd
import yaml
import os


## 1. Repository configuration


In [2]:

from pathlib import Path
import os
import yaml

def find_project_root(start=None):
    """Find the repository root by looking for .git or config.yaml."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() or (candidate / "config.yaml").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()

config_path = PROJECT_ROOT / "config.yaml"
config = {}
if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f) or {}

paths_cfg = config.get("paths", {})

RAW_ROOT = PROJECT_ROOT / paths_cfg.get("raw_dir", "data/raw")
PROCESSED_ROOT = PROJECT_ROOT / paths_cfg.get("processed_dir", "data/processed")

# Optional override for local machines:
# Windows PowerShell example:
#   $env:CYCLING_DATA_DIR="D:\\path\\to\\CYCLING_DATA"
CYCLING_DATA_DIR = Path(
    os.environ.get(
        "CYCLING_DATA_DIR",
        RAW_ROOT / "cycling_data"
    )
).expanduser().resolve()

CYCLING_PROCESSED_DIR = PROCESSED_ROOT / "cycling_counters"
CYCLING_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Cycling raw data:", CYCLING_DATA_DIR)
print("Cycling processed data:", CYCLING_PROCESSED_DIR)


BASE_DIR = CYCLING_DATA_DIR
OUTPUT_DIR = CYCLING_PROCESSED_DIR / "extraction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not BASE_DIR.exists():
    raise FileNotFoundError(
        "Cycling raw-data folder not found. Place yearly folders under "
        "data/raw/cycling_data/ or set CYCLING_DATA_DIR."
    )

print("Extraction outputs:", OUTPUT_DIR)


Project root: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning
Cycling raw data: C:\Users\f_tir\OneDrive\Documents\UNI_2026\CAPSTONE PROJECT\A- TEAM A\CYCLING_DATA
Cycling processed data: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters
Extraction outputs: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\extraction


## 2. Required counters and extraction date ranges

The extraction ranges below include all available years needed for quality checks and possible later analysis.


In [3]:
TARGET_PERIODS = {
    32493: {
        "street": "Wellington Street",
        "start": pd.Timestamp("2019-01-01"),
        "end": pd.Timestamp("2022-12-31"),
    },
    9077: {
        "street": "Albert Street",
        "start": pd.Timestamp("2019-01-01"),
        "end": pd.Timestamp("2022-12-31"),
    },
    34687: {
        "street": "Moorabool Street",
        "start": pd.Timestamp("2020-10-01"),
        "end": pd.Timestamp("2022-12-31"),
    },
    64644: {
        "street": "Moorabool linked ID",
        "start": pd.Timestamp("2020-10-01"),
        "end": pd.Timestamp("2022-12-31"),
    },
    64645: {
        "street": "Moorabool linked ID",
        "start": pd.Timestamp("2020-10-01"),
        "end": pd.Timestamp("2022-12-31"),
    },
    40004: {
        "street": "Heidelberg Road",
        "start": pd.Timestamp("2019-10-01"),
        "end": pd.Timestamp("2022-03-31"),
    },
    40005: {
        "street": "Heidelberg Road",
        "start": pd.Timestamp("2019-10-01"),
        "end": pd.Timestamp("2022-03-31"),
    },
}

TARGET_IDS = set(TARGET_PERIODS)

pd.DataFrame([
    {
        "SITE_ID": site_id,
        "STREET": info["street"],
        "START": info["start"].date(),
        "END": info["end"].date(),
    }
    for site_id, info in TARGET_PERIODS.items()
])


,SITE_ID,STREET,START,END
0,32493,Wellington Street,2019-01-01,2022-12-31
1,9077,Albert Street,2019-01-01,2022-12-31
2,34687,Moorabool Street,2020-10-01,2022-12-31
3,64644,Moorabool linked ID,2020-10-01,2022-12-31
4,64645,Moorabool linked ID,2020-10-01,2022-12-31
5,40004,Heidelberg Road,2019-10-01,2022-03-31
6,40005,Heidelberg Road,2019-10-01,2022-03-31


## 3. Find the daily ZIP files


In [4]:
YEAR_FOLDERS = [
    BASE_DIR / "bicycle_volume_speed_2019",
    BASE_DIR / "bicycle_volume_speed_2020",
    BASE_DIR / "bicycle_volume_speed_2021",
    BASE_DIR / "bicycle_volume_speed_2022",
]

daily_zip_files = []

for folder in YEAR_FOLDERS:
    if folder.exists():
        daily_zip_files.extend(folder.rglob("*.zip"))
    else:
        print("Missing folder:", folder)

daily_zip_files = sorted(daily_zip_files)

print(f"Found {len(daily_zip_files):,} daily ZIP files.")

for path in daily_zip_files[:10]:
    print(path)

if not daily_zip_files:
    raise FileNotFoundError("No daily ZIP files were found.")


Found 437 daily ZIP files.
C:\Users\f_tir\OneDrive\Documents\UNI_2026\CAPSTONE PROJECT\A- TEAM A\CYCLING_DATA\bicycle_volume_speed_2019\2019-01-03_IND.zip
C:\Users\f_tir\OneDrive\Documents\UNI_2026\CAPSTONE PROJECT\A- TEAM A\CYCLING_DATA\bicycle_volume_speed_2019\2019-01-06_IND.zip
C:\Users\f_tir\OneDrive\Documents\UNI_2026\CAPSTONE PROJECT\A- TEAM A\CYCLING_DATA\bicycle_volume_speed_2019\2019-01-07_IND.zip
C:\Users\f_tir\OneDrive\Documents\UNI_2026\CAPSTONE PROJECT\A- TEAM A\CYCLING_DATA\bicycle_volume_speed_2019\2019-01-13_IND.zip
C:\Users\f_tir\OneDrive\Documents\UNI_2026\CAPSTONE PROJECT\A- TEAM A\CYCLING_DATA\bicycle_volume_speed_2019\2019-01-20_IND.zip
C:\Users\f_tir\OneDrive\Documents\UNI_2026\CAPSTONE PROJECT\A- TEAM A\CYCLING_DATA\bicycle_volume_speed_2019\2019-01-27_IND.zip
C:\Users\f_tir\OneDrive\Documents\UNI_2026\CAPSTONE PROJECT\A- TEAM A\CYCLING_DATA\bicycle_volume_speed_2019\2019-02-01_IND.zip
C:\Users\f_tir\OneDrive\Documents\UNI_2026\CAPSTONE PROJECT\A- TEAM A\CYCLING

## 4. Extract only the required counters

The filename pattern recognises IDs even when direction and date text follow the counter number, for example:

```text
IND_D208_X9077_E1_20201005.csv
IND_D208_X32493_NS0_20210111.csv
```

The site ID is then verified inside each CSV before records are kept.


In [5]:
# Improved pattern:
# finds digits immediately after X and stops before "_" or ".csv"
SITE_PATTERN = re.compile(r"_X(\d+)(?=_|\.csv$)", re.IGNORECASE)

def find_site_column(columns):
    normalised = {str(col).strip().upper(): col for col in columns}

    for candidate in [
        "SITE_XN_ROUTE",
        "SITE_XN_R",
        "SITE_XN",
        "SITE_ID",
        "SITE",
    ]:
        if candidate in normalised:
            return normalised[candidate]

    for upper_name, original_name in normalised.items():
        if upper_name.startswith("SITE_XN"):
            return original_name

    return None


matched_frames = []
scan_issues = []
matched_csv_files = 0
candidate_csv_files = 0

for zip_number, zip_path in enumerate(daily_zip_files, start=1):
    try:
        with ZipFile(zip_path, "r") as archive:
            for member in archive.namelist():
                if not member.lower().endswith(".csv"):
                    continue

                filename = Path(member).name
                match = SITE_PATTERN.search(filename)

                # If the filename contains a site ID, skip it unless it is one of our targets.
                if match:
                    filename_site_id = int(match.group(1))

                    if filename_site_id not in TARGET_IDS:
                        continue

                    candidate_csv_files += 1

                # Fallback:
                # If no site ID can be read from the filename, read the CSV
                # and check its site column.
                try:
                    with archive.open(member) as raw_file:
                        df = pd.read_csv(raw_file, low_memory=False)
                except Exception as exc:
                    scan_issues.append({
                        "ZIP": str(zip_path),
                        "CSV": member,
                        "ISSUE": f"CSV read error: {exc}",
                    })
                    continue

                site_column = find_site_column(df.columns)

                if site_column is None:
                    scan_issues.append({
                        "ZIP": str(zip_path),
                        "CSV": member,
                        "ISSUE": "Site ID column not found",
                    })
                    continue

                site_values = pd.to_numeric(
                    df[site_column],
                    errors="coerce",
                )

                wanted_rows = site_values.isin(TARGET_IDS)

                if not wanted_rows.any():
                    continue

                selected = df.loc[wanted_rows].copy()

                selected["SITE_ID"] = pd.to_numeric(
                    selected[site_column],
                    errors="coerce",
                ).astype("Int64")

                selected["SOURCE_ZIP"] = zip_path.name
                selected["SOURCE_CSV"] = filename

                matched_frames.append(selected)
                matched_csv_files += 1

    except BadZipFile as exc:
        scan_issues.append({
            "ZIP": str(zip_path),
            "CSV": "",
            "ISSUE": f"Bad ZIP file: {exc}",
        })

    except Exception as exc:
        scan_issues.append({
            "ZIP": str(zip_path),
            "CSV": "",
            "ISSUE": f"ZIP processing error: {exc}",
        })

    if zip_number % 100 == 0 or zip_number == len(daily_zip_files):
        print(
            f"Scanned {zip_number:,}/{len(daily_zip_files):,} ZIP files"
        )

if not matched_frames:
    raise ValueError(
        "No records were found for any of the requested counters."
    )

raw_target = pd.concat(
    matched_frames,
    ignore_index=True,
)

print()
print(f"Candidate CSV files identified from names: {candidate_csv_files:,}")
print(f"Matched CSV files after content check: {matched_csv_files:,}")
print(f"Raw matching rows: {len(raw_target):,}")
print()
print(
    raw_target["SITE_ID"]
    .value_counts()
    .sort_index()
)

scan_issues_df = pd.DataFrame(scan_issues)

if not scan_issues_df.empty:
    scan_issues_df.to_csv(
        OUTPUT_DIR / "scan_issues.csv",
        index=False,
    )
    print(
        f"Saved {len(scan_issues_df):,} scan issue(s) "
        "to scan_issues.csv"
    )


Scanned 100/437 ZIP files
Scanned 200/437 ZIP files
Scanned 300/437 ZIP files
Scanned 400/437 ZIP files
Scanned 437/437 ZIP files

Candidate CSV files identified from names: 1,085
Matched CSV files after content check: 1,068
Raw matching rows: 2,589,030

SITE_ID
9077     902660
32493    759375
40004    470485
40005    456510
Name: count, dtype: Int64


## 5. Confirm which requested counters were found


In [6]:
found_ids = set(
    raw_target["SITE_ID"]
    .dropna()
    .astype(int)
    .unique()
)

missing_target_ids = TARGET_IDS - found_ids

print("Found target counters:", sorted(found_ids))

if missing_target_ids:
    print("Counters not found in measurement data:", sorted(missing_target_ids))
else:
    print("All requested counters were found.")


Found target counters: [np.int64(9077), np.int64(32493), np.int64(40004), np.int64(40005)]
Counters not found in measurement data: [34687, 64644, 64645]


## 6. Parse dates and apply the exact required periods


In [7]:
def find_column(columns, wanted_name):
    normalised = {
        str(col).strip().upper(): col
        for col in columns
    }

    wanted_upper = wanted_name.upper()

    if wanted_upper in normalised:
        return normalised[wanted_upper]

    for upper_name, original_name in normalised.items():
        if wanted_upper in upper_name:
            return original_name

    return None


date_column = find_column(
    raw_target.columns,
    "DATE",
)

speed_column = find_column(
    raw_target.columns,
    "SPEED",
)

if date_column is None:
    raise ValueError("DATE column not found.")

raw_target["DATE_PARSED"] = pd.to_datetime(
    raw_target[date_column],
    errors="coerce",
    dayfirst=True,
).dt.normalize()

print("Date column:", date_column)
print("Speed column:", speed_column)
print(
    "Unparseable date values:",
    raw_target["DATE_PARSED"].isna().sum(),
)

filtered_parts = []

for site_id, info in TARGET_PERIODS.items():
    part = raw_target.loc[
        raw_target["SITE_ID"] == site_id
    ].copy()

    part = part.loc[
        part["DATE_PARSED"].between(
            info["start"],
            info["end"],
            inclusive="both",
        )
    ].copy()

    part["STREET"] = info["street"]

    filtered_parts.append(part)

filtered = pd.concat(
    filtered_parts,
    ignore_index=True,
)

print()
print(f"Rows after date filtering: {len(filtered):,}")

print(
    filtered.groupby("SITE_ID")["DATE_PARSED"]
    .agg(["min", "max", "count"])
)


Date column: DATE
Speed column: SPEED
Unparseable date values: 0

Rows after date filtering: 2,312,030
               min        max   count
SITE_ID                              
9077    2019-01-01 2022-12-25  902230
32493   2019-01-01 2022-12-25  758944
40004   2021-01-09 2022-03-31  323296
40005   2021-01-09 2022-03-31  327560


## 5A. Create a counter availability summary

This uses the actual dates stored inside the CSV records.


In [8]:
availability_rows = []

for site_id, info in TARGET_PERIODS.items():
    site_raw = raw_target.loc[
        raw_target["SITE_ID"] == site_id
    ].copy()

    if site_raw.empty:
        availability_rows.append({
            "SITE_ID": site_id,
            "STREET": info["street"],
            "FOUND": "No",
            "FIRST_AVAILABLE_DATE": pd.NaT,
            "LAST_AVAILABLE_DATE": pd.NaT,
            "TOTAL_RAW_ROWS": 0,
        })
    else:
        availability_rows.append({
            "SITE_ID": site_id,
            "STREET": info["street"],
            "FOUND": "Yes",
            "FIRST_AVAILABLE_DATE": site_raw["DATE_PARSED"].min(),
            "LAST_AVAILABLE_DATE": site_raw["DATE_PARSED"].max(),
            "TOTAL_RAW_ROWS": len(site_raw),
        })

availability_summary = pd.DataFrame(availability_rows)

availability_summary.to_csv(
    OUTPUT_DIR / "counter_availability_summary.csv",
    index=False,
)

availability_summary


,SITE_ID,STREET,FOUND,FIRST_AVAILABLE_DATE,LAST_AVAILABLE_DATE,TOTAL_RAW_ROWS
0,32493,Wellington Street,Yes,2018-12-31,2022-12-25,759375
1,9077,Albert Street,Yes,2018-12-31,2022-12-25,902660
2,34687,Moorabool Street,No,NaT,NaT,0
3,64644,Moorabool linked ID,No,NaT,NaT,0
4,64645,Moorabool linked ID,No,NaT,NaT,0
5,40004,Heidelberg Road,Yes,2021-01-09,2022-12-18,470485
6,40005,Heidelberg Road,Yes,2021-01-09,2022-12-18,456510


## 7. Save combined and counter-specific raw files


In [9]:
combined_path = (
    OUTPUT_DIR
    / "cycling_required_counters_raw.csv"
)

filtered.to_csv(
    combined_path,
    index=False,
)

print("Saved:", combined_path)

for site_id in sorted(TARGET_IDS):
    counter_path = (
        OUTPUT_DIR
        / f"counter_{site_id}_raw.csv"
    )

    filtered.loc[
        filtered["SITE_ID"] == site_id
    ].to_csv(
        counter_path,
        index=False,
    )

    print("Saved:", counter_path)


Saved: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\extraction\cycling_required_counters_raw.csv
Saved: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\extraction\counter_9077_raw.csv
Saved: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\extraction\counter_32493_raw.csv
Saved: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\extraction\counter_34687_raw.csv
Saved: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\extraction\counter_40004_raw.csv
Saved: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\extraction\counter_40005_raw.csv
Saved: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\extraction\counter_64644_raw.csv
Saved: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\extraction\cou

## 8. Coverage and missing-date checks

A missing date can mean:

- genuinely zero detected cyclists, or
- a collection/data gap.

The notebook flags these dates for review rather than assuming they are true zero-volume days.


In [10]:
coverage_rows = []
missing_date_rows = []

for site_id, info in TARGET_PERIODS.items():
    site_data = filtered.loc[
        filtered["SITE_ID"] == site_id
    ]

    observed_dates = pd.DatetimeIndex(
        site_data["DATE_PARSED"]
        .dropna()
        .unique()
    ).normalize()

    expected_dates = pd.date_range(
        info["start"],
        info["end"],
        freq="D",
    )

    missing_dates = expected_dates.difference(
        observed_dates
    )

    coverage_rows.append({
        "SITE_ID": site_id,
        "STREET": info["street"],
        "REQUIRED_START": info["start"].date(),
        "REQUIRED_END": info["end"].date(),
        "EXPECTED_DAYS": len(expected_dates),
        "DAYS_WITH_RECORDS": len(observed_dates),
        "MISSING_OR_ZERO_OBSERVATION_DAYS": len(missing_dates),
        "COVERAGE_PERCENT": round(
            100
            * len(observed_dates)
            / len(expected_dates),
            2,
        ),
    })

    for missing_date in missing_dates:
        missing_date_rows.append({
            "SITE_ID": site_id,
            "STREET": info["street"],
            "DATE": missing_date.date(),
        })

coverage_summary = pd.DataFrame(
    coverage_rows
)

missing_dates_df = pd.DataFrame(
    missing_date_rows
)

coverage_summary.to_csv(
    OUTPUT_DIR / "coverage_summary.csv",
    index=False,
)

missing_dates_df.to_csv(
    OUTPUT_DIR
    / "missing_dates_or_zero_observation_days.csv",
    index=False,
)

coverage_summary


,SITE_ID,STREET,REQUIRED_START,REQUIRED_END,EXPECTED_DAYS,DAYS_WITH_RECORDS,MISSING_OR_ZERO_OBSERVATION_DAYS,COVERAGE_PERCENT
0,32493,Wellington Street,2019-01-01,2022-12-31,1461,1455,6,99.59
1,9077,Albert Street,2019-01-01,2022-12-31,1461,1287,174,88.09
2,34687,Moorabool Street,2020-10-01,2022-12-31,822,0,822,0.00
3,64644,Moorabool linked ID,2020-10-01,2022-12-31,822,0,822,0.00
4,64645,Moorabool linked ID,2020-10-01,2022-12-31,822,0,822,0.00
5,40004,Heidelberg Road,2019-10-01,2022-03-31,913,412,501,45.13
6,40005,Heidelberg Road,2019-10-01,2022-03-31,913,412,501,45.13


## 9. Continuous collection-gap periods


In [11]:
gap_rows = []

if not missing_dates_df.empty:
    for site_id, group in missing_dates_df.groupby("SITE_ID"):
        dates = pd.to_datetime(
            group["DATE"]
        ).sort_values().reset_index(drop=True)

        gap_start = dates.iloc[0]
        previous_date = dates.iloc[0]

        for current_date in dates.iloc[1:]:
            if (current_date - previous_date).days == 1:
                previous_date = current_date
                continue

            gap_rows.append({
                "SITE_ID": site_id,
                "STREET": TARGET_PERIODS[int(site_id)]["street"],
                "GAP_START": gap_start.date(),
                "GAP_END": previous_date.date(),
                "GAP_DAYS": (
                    previous_date - gap_start
                ).days + 1,
            })

            gap_start = current_date
            previous_date = current_date

        gap_rows.append({
            "SITE_ID": site_id,
            "STREET": TARGET_PERIODS[int(site_id)]["street"],
            "GAP_START": gap_start.date(),
            "GAP_END": previous_date.date(),
            "GAP_DAYS": (
                previous_date - gap_start
            ).days + 1,
        })

collection_gaps = pd.DataFrame(
    gap_rows
)

if collection_gaps.empty:
    print("No missing-date gaps detected.")
else:
    collection_gaps = collection_gaps.sort_values(
        ["SITE_ID", "GAP_START"]
    )

    collection_gaps.to_csv(
        OUTPUT_DIR / "collection_gaps.csv",
        index=False,
    )

    print(
        collection_gaps.to_string(
            index=False
        )
    )


 SITE_ID              STREET  GAP_START    GAP_END  GAP_DAYS
    9077       Albert Street 2019-08-05 2020-01-19       168
    9077       Albert Street 2022-12-26 2022-12-31         6
   32493   Wellington Street 2022-12-26 2022-12-31         6
   34687    Moorabool Street 2020-10-01 2022-12-31       822
   40004     Heidelberg Road 2019-10-01 2021-01-08       466
   40004     Heidelberg Road 2021-12-13 2021-12-19         7
   40004     Heidelberg Road 2022-01-03 2022-01-09         7
   40004     Heidelberg Road 2022-02-07 2022-02-20        14
   40004     Heidelberg Road 2022-02-28 2022-03-06         7
   40005     Heidelberg Road 2019-10-01 2021-01-08       466
   40005     Heidelberg Road 2021-12-13 2021-12-19         7
   40005     Heidelberg Road 2022-01-03 2022-01-09         7
   40005     Heidelberg Road 2022-02-07 2022-02-20        14
   40005     Heidelberg Road 2022-02-28 2022-03-06         7
   64644 Moorabool linked ID 2020-10-01 2022-12-31       822
   64645 Moorabool linke

## 10. Duplicate-record check


In [12]:
helper_columns = {
    "SOURCE_ZIP",
    "SOURCE_CSV",
    "SITE_ID",
    "STREET",
    "DATE_PARSED",
}

measurement_columns = [
    column
    for column in filtered.columns
    if column not in helper_columns
]

duplicate_mask = filtered.duplicated(
    subset=measurement_columns,
    keep=False,
)

duplicate_records = filtered.loc[
    duplicate_mask
].copy()

print(
    f"Rows involved in duplicate groups: "
    f"{len(duplicate_records):,}"
)

if not duplicate_records.empty:
    duplicate_records.to_csv(
        OUTPUT_DIR / "duplicate_records.csv",
        index=False,
    )


Rows involved in duplicate groups: 39,685


## 11. Zero-speed check


In [13]:
if speed_column is None:
    print("No SPEED column found.")
else:
    speed_numeric = pd.to_numeric(
        filtered[speed_column],
        errors="coerce",
    )

    zero_speed_records = filtered.loc[
        speed_numeric.eq(0)
    ].copy()

    print(
        f"Rows with SPEED = 0: "
        f"{len(zero_speed_records):,}"
    )

    if not zero_speed_records.empty:
        zero_speed_records.to_csv(
            OUTPUT_DIR / "zero_speed_records.csv",
            index=False,
        )


Rows with SPEED = 0: 0


## 12. Daily bicycle volume

Each row represents one bicycle detection, so daily volume is the number of rows per counter per day.


In [14]:
daily_volume = (
    filtered
    .groupby(
        ["SITE_ID", "STREET", "DATE_PARSED"]
    )
    .size()
    .rename("BICYCLE_DETECTIONS")
    .reset_index()
    .sort_values(
        ["SITE_ID", "DATE_PARSED"]
    )
)

daily_volume.to_csv(
    OUTPUT_DIR / "daily_bicycle_volume.csv",
    index=False,
)

daily_volume.head(20)


,SITE_ID,STREET,DATE_PARSED,BICYCLE_DETECTIONS
0,9077,Albert Street,2019-01-01,195
1,9077,Albert Street,2019-01-02,806
2,9077,Albert Street,2019-01-03,975
3,9077,Albert Street,2019-01-04,558
4,9077,Albert Street,2019-01-05,303
5,9077,Albert Street,2019-01-06,334
6,9077,Albert Street,2019-01-07,1395
7,9077,Albert Street,2019-01-08,1592
8,9077,Albert Street,2019-01-09,1471
9,9077,Albert Street,2019-01-10,1574


## 13. Daily speed summary


In [15]:
if speed_column is None:
    print("No SPEED column found.")
else:
    speed_data = filtered.copy()

    speed_data["SPEED_NUMERIC"] = pd.to_numeric(
        speed_data[speed_column],
        errors="coerce",
    )

    daily_speed_summary = (
        speed_data
        .groupby(
            ["SITE_ID", "STREET", "DATE_PARSED"]
        )["SPEED_NUMERIC"]
        .agg(
            BICYCLE_DETECTIONS="count",
            MEAN_SPEED="mean",
            MEDIAN_SPEED="median",
            MIN_SPEED="min",
            MAX_SPEED="max",
        )
        .reset_index()
        .sort_values(
            ["SITE_ID", "DATE_PARSED"]
        )
    )

    daily_speed_summary.to_csv(
        OUTPUT_DIR / "daily_speed_summary.csv",
        index=False,
    )

    daily_speed_summary.head(20)


# Output location

After running all cells, open:

```text
CYCLING_DATA/data/processed/cycling_counters/extraction/
```

Important outputs include:

- `cycling_required_counters_raw.csv`
- one `counter_<ID>_raw.csv` file for each requested counter,
- `coverage_summary.csv`
- `missing_dates_or_zero_observation_days.csv`
- `collection_gaps.csv`
- `duplicate_records.csv` when duplicates are flagged,
- `zero_speed_records.csv` when zero-speed records exist,
- `daily_bicycle_volume.csv`
- `daily_speed_summary.csv`
- `counter_availability_summary.csv`.
